# Advanced Text Preprocessing Pipeline for IMDB Reviews

I decided to upgrade the previous pipeline. To get as close to perfect accuracy as possible, we need to handle the really difficult parts of language: negations, contractions, typos, and proper part-of-speech lemmatization.
Here is the ultimate, state-of-the-art preprocessing pipeline.

In [36]:
!pip install emoji tqdm nltk pandas contractions symspellpy setuptools 


### Step 1: Loading Data & Setup

In [37]:
import pandas as pd
import re
import string
import nltk
import emoji
import contractions
from collections import Counter
from tqdm import tqdm
import importlib.resources
from symspellpy import SymSpell, Verbosity

pd.set_option('display.max_colwidth', None)
tqdm.pandas()

print("Loading dataset...")
df = pd.read_csv('../Data/Sentiment_Analysis/IMDB Dataset.csv', encoding='latin-1')
print(f"Loaded {df.shape[0]} reviews.")

Loading dataset...
Loaded 50000 reviews.


### Step 2: Lowercasing & Basic HTML Removal

In [38]:
df['review'] = df['review'].str.lower()

def remove_html_and_urls(text):
    text = re.sub('<.*?>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return text

df['review'] = df['review'].apply(remove_html_and_urls)

### Step 3: Expanding Contractions
Before removing punctuation, I need to expand contractions (e.g. "don't" -> "do not"). This is crucial so "not" can be processed later.

In [39]:
def expand_contractions(text):
    return contractions.fix(text)

df['review'] = df['review'].apply(expand_contractions)

### Step 4: Number Replacement & Emoji Removal
Numbers don't usually imply sentiment, so I'll replace them with a special `<NUM>` token.

In [40]:
def replace_numbers(text):
    return re.sub(r'\b\d+\b', '<NUM>', text)

def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

df['review'] = df['review'].apply(replace_numbers).apply(remove_emoji)

### Step 5: Removing Punctuation (Except Negation Indicators)
I will remove all punctuation.

In [41]:
def remove_punc(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['review'] = df['review'].apply(remove_punc)

### Step 6: Spelling Correction (SymSpell)
I'm using `symspellpy` because it is orders of magnitude faster than TextBlob. This will fix obvious typos.

In [42]:
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = str(importlib.resources.files(
    "symspellpy") / "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

def correct_spelling(text):
    words = text.split()
    corrected = []
    for w in words:
        if w == '<NUM>':
            corrected.append(w)
            continue
        suggestions = sym_spell.lookup(w, Verbosity.CLOSEST, max_edit_distance=2, include_unknown=True)
        if suggestions:
            corrected.append(suggestions[0].term)
        else:
            corrected.append(w)
    return " ".join(corrected)

print("Correcting spelling... ")
df['review'] = df['review'].progress_apply(correct_spelling)

Correcting spelling... (This may take a few minutes)


100%|██████████| 50000/50000 [02:06<00:00, 394.71it/s]


### Step 7: Negation Handling
I am writing a custom function to join "not", "no", "never" with the next word. E.g., "not good" -> "not_good".

In [43]:
def handle_negation(text):
    words = text.split()
    negation_words = {'not', 'no', 'never'}
    new_words = []
    i = 0
    while i < len(words):
        if words[i] in negation_words and i+1 < len(words):
            new_words.append(words[i] + "_" + words[i+1])
            i += 2
        else:
            new_words.append(words[i])
            i += 1
    return " ".join(new_words)

df['review'] = df['review'].apply(handle_negation)

### Step 8: Stopword Removal
Standard stopword removal, but ensuring we don't accidentally remove our newly created negated words.

In [44]:
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words = stop_words - {'not', 'no', 'never'}

def remove_stopwords(text):
    words = text.split()
    return " ".join([w for w in words if w not in stop_words])

df['review'] = df['review'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\raghuvaran\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Step 9: POS Tagging & Lemmatization
Instead of blind lemmatization, I'm finding the Part-of-Speech for each word so the lemmatizer knows exactly how to cut it down.

In [45]:
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def pos_lemmatize(text):
    words = text.split()
    # Temporarily remove underscore for tagging
    pos_tags = nltk.pos_tag([w.replace('_', ' ') for w in words])
    
    lemmatized = []
    for i, (word, tag) in enumerate(pos_tags):
        original_word = words[i]
        # Don't lemmatize negated compounds
        if '_' in original_word or original_word == '<NUM>':
            lemmatized.append(original_word)
        else:
            lemmatized.append(lemmatizer.lemmatize(word, get_wordnet_pos(tag)))
    return " ".join(lemmatized)

print("Lemmatizing with POS tags...")
df['review'] = df['review'].progress_apply(pos_lemmatize)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\raghuvaran\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\raghuvaran\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Lemmatizing with POS tags...


100%|██████████| 50000/50000 [07:23<00:00, 112.85it/s]


### Step 10: Removing Rare and Highly Frequent Words
Finally, I'm stripping out words that only appear once in the whole dataset, and words that appear in over 90% of reviews.

In [46]:
print("Counting frequencies...")
all_words = " ".join(df['review']).split()
word_counts = Counter(all_words)

total_docs = len(df)
rare_words = set([w for w, c in word_counts.items() if c == 1])
freq_words = set([w for w, c in word_counts.items() if c > (0.9 * total_docs)])

words_to_remove = rare_words.union(freq_words)

def remove_extreme_words(text):
    return " ".join([w for w in text.split() if w not in words_to_remove])

print("Removing extreme words...")
df['review'] = df['review'].progress_apply(remove_extreme_words)

Counting frequencies...
Removing extreme words...


100%|██████████| 50000/50000 [00:02<00:00, 19935.83it/s]


### Step 11: Export
Done! Exporting the pristine dataset.

In [47]:
df.to_csv('../Data/Sentiment_Analysis/IMDB_Dataset_Cleaned_Advanced.csv', index=False)
print("Dataset exported successfully!")

Dataset exported successfully!
